# quantbench_market_regime_analysis_v1

目标：把 `research/quantbench_all_model_run_ledger_v1/daily_backtest.csv` 的 32 个 QuantBench 模型结果，接入 `src.analysis.build_market_regime_analysis_report`，使用真实 SOXL/SOXX/VIX market panel 重新生成 market-regime/window 模型差异分析。

外部参考：Qlib 官方 report/analysis 文档覆盖 portfolio cumulative return、excess return、turnover、risk analysis 等分析对象。本仓库不直接暴露 qlib recorder 内部路径，而是把结果标准化成 manifest-first CSV/JSON。

- Qlib GitHub: https://github.com/microsoft/qlib
- Qlib report docs: https://qlib.readthedocs.io/en/latest/component/report.html

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.analysis import (
    MarketRegimeConfig,
    build_analysis_windows,
    build_market_regime_analysis_report,
    write_analysis_report,
)
from src.core import DatasetSnapshot
from src.data import load_soxl_support_frame

CARD = REPO_ROOT / 'research' / 'quantbench_market_regime_analysis_v1'
SOURCE_LEDGER = REPO_ROOT / 'research' / 'quantbench_all_model_run_ledger_v1'
REPORT_DIR = CARD / 'analysis_report'
CARD.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


## 1. 读取 QuantBench 全模型 ledger

In [2]:
daily = pd.read_csv(SOURCE_LEDGER / 'daily_backtest.csv', parse_dates=['date'])
source_summary = json.loads((SOURCE_LEDGER / 'model_run_ledger_summary.json').read_text(encoding='utf-8'))
source_manifest = json.loads((SOURCE_LEDGER / 'analysis_manifest.json').read_text(encoding='utf-8'))
source_comparison = pd.read_csv(SOURCE_LEDGER / 'model_comparison.csv')

start = str(daily['date'].min().date())
end = str(daily['date'].max().date())
baseline_model = str(source_comparison.sort_values('rank').iloc[0]['model'])

source_overview = {
    'daily_rows': int(len(daily)),
    'model_count': int(daily['model'].nunique()),
    'start': start,
    'end': end,
    'baseline_model': baseline_model,
    'native_model_count': int(source_summary['native_model_count']),
    'proxy_model_count': int(source_summary['proxy_model_count']),
}
source_overview


{'daily_rows': 832,
 'model_count': 32,
 'start': '2026-05-01',
 'end': '2026-06-08',
 'baseline_model': 'mlp_mixer_mse',
 'native_model_count': 28,
 'proxy_model_count': 4}

## 2. 用真实市场面板生成 market regime report

In [3]:
market = load_soxl_support_frame(start=start, end=end)
windows = build_analysis_windows(
    start=start,
    end=end,
    custom_windows={
        'quantbench_test': (start, end),
        'first_half': (start, str(daily['date'].sort_values().drop_duplicates().iloc[len(daily['date'].drop_duplicates()) // 2].date())),
        'second_half': (str(daily['date'].sort_values().drop_duplicates().iloc[len(daily['date'].drop_duplicates()) // 2].date()), end),
    },
)
config = MarketRegimeConfig(
    trend_window=21,
    volatility_window=21,
    drawdown_window=63,
    volume_window=20,
    high_vix_threshold=30.0,
    low_vix_threshold=20.0,
    stress_drawdown_threshold=-0.10,
    bear_drawdown_threshold=-0.03,
    uptrend_threshold=0.03,
    downtrend_threshold=-0.03,
    sideways_trend_abs_threshold=0.01,
    high_volatility_threshold=0.045,
    low_volatility_threshold=0.020,
    volume_shock_z_threshold=2.0,
)

dataset = DatasetSnapshot(
    name='quantbench_market_regime_analysis_v1',
    universe=('SOXL', 'SOXX', 'VIX'),
    start=start,
    end=end,
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(
        'research/quantbench_all_model_run_ledger_v1/daily_backtest.csv',
        'data/processed/panels/soxl_soxx_vix_full.csv',
    ),
    metadata={
        'source_quantbench_manifest_hash': source_manifest.get('manifest_hash', ''),
        'source_model_count': source_overview['model_count'],
        'source_native_model_count': source_overview['native_model_count'],
        'source_proxy_model_count': source_overview['proxy_model_count'],
    },
)

report = build_market_regime_analysis_report(
    artifact_id='quantbench_market_regime_analysis_v1',
    daily_backtest=daily,
    market_frame=market,
    windows=windows,
    baseline_model=baseline_model,
    config=config,
    summary={
        'analysis_direction': 'analysis',
        'source_ledger': 'research/quantbench_all_model_run_ledger_v1',
        'source_manifest_hash': source_manifest.get('manifest_hash', ''),
        'source_model_run_ledger_summary': source_summary,
        'qlib_reference': {
            'github': 'https://github.com/microsoft/qlib',
            'report_docs': 'https://qlib.readthedocs.io/en/latest/component/report.html',
            'usage': 'Qlib report/risk-analysis objects are used as external analysis references; SOXL regime slicing is implemented in src.analysis.',
        },
        'note': '中文 review：使用真实 market regime 重算 QuantBench 全模型分段差异。',
    },
    metadata={
        'source_type': 'quantbench_daily_backtest_plus_market_regime',
        'source_ledger': 'research/quantbench_all_model_run_ledger_v1',
    },
)
export = write_analysis_report(report, REPORT_DIR, dataset=dataset)

# Keep top-level sidecars for quick review.
report.tables['market_regimes'].to_csv(CARD / 'market_regimes.csv', index=False)
report.tables['market_regime_counts'].to_csv(CARD / 'market_regime_counts.csv', index=False)
report.tables['market_regime_daily_backtest'].to_csv(CARD / 'market_regime_daily_backtest.csv', index=False)

result_overview = {
    'schema_version': 'research.quantbench_market_regime_analysis_v1',
    'status': 'completed',
    'source_ledger': 'research/quantbench_all_model_run_ledger_v1',
    'source_manifest_hash': source_manifest.get('manifest_hash', ''),
    'start': start,
    'end': end,
    'baseline_model': baseline_model,
    'source_model_count': source_overview['model_count'],
    'native_model_count': source_overview['native_model_count'],
    'proxy_model_count': source_overview['proxy_model_count'],
    'daily_rows': source_overview['daily_rows'],
    'market_regime_rows': int(len(report.tables['market_regimes'])),
    'market_regime_count': int(report.summary['market_regime_count']),
    'market_regime_counts': report.tables['market_regime_counts'].to_dict(orient='records'),
    'window_count': int(len(windows)),
    'analysis_report': 'research/quantbench_market_regime_analysis_v1/analysis_report',
    'analysis_table_counts': report.summary['analysis_table_counts'],
    'top_weakness': report.summary.get('analysis_overview', {}).get('top_weakness', {}),
    'top_strength': report.summary.get('analysis_overview', {}).get('top_strength', {}),
    'top_regime_fragile_models': report.summary.get('analysis_overview', {}).get('top_regime_fragile_models', [])[:5],
    'export': export,
}
(CARD / 'analysis_summary.json').write_text(json.dumps(result_overview, ensure_ascii=False, indent=2), encoding='utf-8')
result_overview


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.quantbench_market_regime_analysis_v1',
 'status': 'completed',
 'source_ledger': 'research/quantbench_all_model_run_ledger_v1',
 'source_manifest_hash': '77e70bdef56f9e6bb2bc2212df3a00ae50b82c736f894556be15ee415e8d3aea',
 'start': '2026-05-01',
 'end': '2026-06-08',
 'baseline_model': 'mlp_mixer_mse',
 'source_model_count': 32,
 'native_model_count': 28,
 'proxy_model_count': 4,
 'daily_rows': 832,
 'market_regime_rows': 26,
 'market_regime_count': 4,
 'market_regime_counts': [{'regime': 'high_vol_uptrend',
   'row_count': 19,
   'row_share': 0.7307692307692307},
  {'regime': 'stress', 'row_count': 5, 'row_share': 0.19230769230769232},
  {'regime': 'neutral', 'row_count': 1, 'row_share': 0.038461538461538464},
  {'regime': 'sideways', 'row_count': 1, 'row_share': 0.038461538461538464}],
 'window_count': 8,
 'analysis_report': 'research/quantbench_market_regime_analysis_v1/analysis_report',
 'analysis_table_counts': {'market_regimes': 26,
  'market_regime_co

## 3. 关键对比表

In [4]:
comparison = pd.read_csv(REPORT_DIR / 'model_comparison.csv')
segment_matrix = pd.read_csv(REPORT_DIR / 'segment_effect_matrix.csv')
briefing = pd.read_csv(REPORT_DIR / 'model_review_briefing.csv')
diagnostics = pd.read_csv(REPORT_DIR / 'model_regime_window_diagnostics.csv')

print('Top models by market-regime reanalysis:')
display(comparison.head(8))
print('Largest segment spreads:')
display(segment_matrix.sort_values('relative_return_pct_spread', ascending=False).head(8))
print('Priority briefing:')
display(briefing.head(8))
print('Fragile diagnostics:')
display(diagnostics.sort_values(['weakness_segment_count', 'stress_negative_count'], ascending=False).head(8))


Top models by market-regime reanalysis:


,rank,model,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,1,mlp_mixer_mse,55.285791,58.118381,-2.832590,20.475703,4.121763,0.763458,5.091698,60.725394
1,2,rsr_mse,45.636395,58.118381,-12.481986,22.981182,3.683452,0.747728,5.036027,43.672385
2,3,tcn_mse,44.126973,58.118381,-13.991408,24.842772,3.516625,0.762473,3.515777,41.208206
3,4,mlp_mse,45.180246,58.118381,-12.938135,24.485225,3.757595,0.702538,6.004026,40.141611
4,5,fedformer_mse,19.532160,58.118381,-38.586221,7.586366,3.648972,0.278940,4.978693,36.581539
5,6,darnn_mse,37.592797,58.118381,-20.525584,26.828884,3.161380,0.725121,0.787754,34.094988
6,7,thgnn_mse,44.019103,58.118381,-14.099279,28.171913,3.439516,0.755381,4.754980,33.689408
7,8,hawkes_gru_mse,31.407994,58.118381,-26.710387,14.707499,3.249848,0.643984,6.476610,32.568884


Largest segment spreads:


,schema_version,segment_type,segment,window,regime,baseline_model,model_count,best_relative_return_model,best_relative_return_pct,worst_relative_return_model,...,best_stability_vs_baseline_delta,highest_drawdown_model,highest_drawdown_pct,lowest_drawdown_model,lowest_drawdown_pct,drawdown_pct_spread,highest_turnover_model,highest_turnover,average_turnover,review_note
0,analysis.segment_effect_matrix.v1,all,all,NaN,NaN,mlp_mixer_mse,32,mlp_mixer_mse,-2.832590,ridge_mse,...,0.000000,sthan_mse,34.937623,fedformer_mse,7.586366,27.351257,sthgcn_mse,8.34466,4.157516,mlp_mixer_mse 在 all 领先；ridge_mse 明显弱于基准，需检查该分段适配。
10,analysis.segment_effect_matrix.v1,window,trailing_3m,trailing_3m,NaN,mlp_mixer_mse,32,mlp_mixer_mse,-2.832590,ridge_mse,...,0.000000,sthan_mse,34.937623,fedformer_mse,7.586366,27.351257,sthgcn_mse,8.34466,4.157516,mlp_mixer_mse 在 trailing_3m 领先；ridge_mse 明显弱于基...
6,analysis.segment_effect_matrix.v1,window,quantbench_test,quantbench_test,NaN,mlp_mixer_mse,32,mlp_mixer_mse,-2.832590,ridge_mse,...,0.000000,sthan_mse,34.937623,fedformer_mse,7.586366,27.351257,sthgcn_mse,8.34466,4.157516,mlp_mixer_mse 在 quantbench_test 领先；ridge_mse 明...
12,analysis.segment_effect_matrix.v1,window,ytd,ytd,NaN,mlp_mixer_mse,32,mlp_mixer_mse,-2.832590,ridge_mse,...,0.000000,sthan_mse,34.937623,fedformer_mse,7.586366,27.351257,sthgcn_mse,8.34466,4.157516,mlp_mixer_mse 在 ytd 领先；ridge_mse 明显弱于基准，需检查该分段适配。
8,analysis.segment_effect_matrix.v1,window,trailing_12m,trailing_12m,NaN,mlp_mixer_mse,32,mlp_mixer_mse,-2.832590,ridge_mse,...,0.000000,sthan_mse,34.937623,fedformer_mse,7.586366,27.351257,sthgcn_mse,8.34466,4.157516,mlp_mixer_mse 在 trailing_12m 领先；ridge_mse 明显弱于...
11,analysis.segment_effect_matrix.v1,window,trailing_6m,trailing_6m,NaN,mlp_mixer_mse,32,mlp_mixer_mse,-2.832590,ridge_mse,...,0.000000,sthan_mse,34.937623,fedformer_mse,7.586366,27.351257,sthgcn_mse,8.34466,4.157516,mlp_mixer_mse 在 trailing_6m 领先；ridge_mse 明显弱于基...
36,analysis.segment_effect_matrix.v1,window_regime,trailing_6m::stress,trailing_6m,stress,mlp_mixer_mse,32,mlp_mixer_mse,7.610536,ridge_ic,...,13.003543,market_graph_mse,9.218404,autoformer_mse,0.002890,9.215514,ridge_mse,2.09264,0.894864,mlp_mixer_mse 在 trailing_6m::stress 领先；ridge_i...
32,analysis.segment_effect_matrix.v1,window_regime,trailing_3m::stress,trailing_3m,stress,mlp_mixer_mse,32,mlp_mixer_mse,7.610536,ridge_ic,...,13.003543,market_graph_mse,9.218404,autoformer_mse,0.002890,9.215514,ridge_mse,2.09264,0.894864,mlp_mixer_mse 在 trailing_3m::stress 领先；ridge_i...


Priority briefing:


,schema_version,priority_rank,model,baseline_model,segment_type,segment,window,regime,review_signal,decision_hint,priority_score,relative_return_pct_delta,stability_score_delta,max_drawdown_pct_delta,turnover_delta,diagnosis,review_note
0,analysis.model_review_briefing.v1,1,ridge_ic,mlp_mixer_mse,regime,stress,NaN,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 stress 弱于基准 mlp_mixer_mse：相对收益和稳定性同...
1,analysis.model_review_briefing.v1,2,ridge_ic,mlp_mixer_mse,window_regime,quantbench_test::stress,quantbench_test,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 quantbench_test::stress 弱于基准 mlp_mi...
2,analysis.model_review_briefing.v1,3,ridge_ic,mlp_mixer_mse,window_regime,trailing_12m::stress,trailing_12m,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 trailing_12m::stress 弱于基准 mlp_mixer...
3,analysis.model_review_briefing.v1,4,ridge_ic,mlp_mixer_mse,window_regime,trailing_1m::stress,trailing_1m,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 trailing_1m::stress 弱于基准 mlp_mixer_...
4,analysis.model_review_briefing.v1,5,ridge_ic,mlp_mixer_mse,window_regime,trailing_3m::stress,trailing_3m,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 trailing_3m::stress 弱于基准 mlp_mixer_...
5,analysis.model_review_briefing.v1,6,ridge_ic,mlp_mixer_mse,window_regime,trailing_6m::stress,trailing_6m,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 trailing_6m::stress 弱于基准 mlp_mixer_...
6,analysis.model_review_briefing.v1,7,ridge_ic,mlp_mixer_mse,window_regime,ytd::stress,ytd,stress,weakness,block_or_rework,288.094874,-41.793206,-204.445191,-1.382083,0.632710,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_ic 在 ytd::stress 弱于基准 mlp_mixer_mse：相对收益...
7,analysis.model_review_briefing.v1,8,ridge_mse,mlp_mixer_mse,regime,stress,NaN,stress,weakness,block_or_rework,254.288771,-41.011654,-172.161971,-0.424896,1.034909,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,ridge_mse 在 stress 弱于基准 mlp_mixer_mse：相对收益和稳定性...


Fragile diagnostics:


,schema_version,model,baseline_model,evaluated_segment_count,positive_segment_count,negative_segment_count,weakness_segment_count,strength_segment_count,risk_segment_count,window_regime_negative_count,...,best_stability_score_delta,worst_segment_type,worst_segment,worst_window,worst_regime,worst_relative_return_pct_delta,worst_stability_score_delta,worst_max_drawdown_pct_delta,diagnostic_signal,review_note
3,analysis.model_regime_window_diagnostic.v1,catboost_rank,mlp_mixer_mse,41,0,27,27,0,16,16,...,0.621817,all,all,NaN,NaN,-39.061302,-71.040439,7.331050,stress_fragile,catboost_rank 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱分段为...
4,analysis.model_regime_window_diagnostic.v1,extra_trees_rank,mlp_mixer_mse,41,0,27,27,0,16,16,...,0.673909,all,all,NaN,NaN,-23.597602,-39.911202,4.576717,stress_fragile,extra_trees_rank 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱...
5,analysis.model_regime_window_diagnostic.v1,xgboost_mse,mlp_mixer_mse,41,0,27,27,0,16,16,...,0.957461,all,all,NaN,NaN,-28.041109,-54.431965,8.943279,stress_fragile,xgboost_mse 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱分段为 a...
21,analysis.model_regime_window_diagnostic.v1,sthgcn_mse,mlp_mixer_mse,41,0,27,27,0,0,16,...,0.000000,all,all,NaN,NaN,-41.496254,-68.182620,-1.971665,stress_fragile,sthgcn_mse 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱分段为 al...
1,analysis.model_regime_window_diagnostic.v1,gcn_mse,mlp_mixer_mse,41,1,26,26,1,23,15,...,24.946642,all,all,NaN,NaN,-44.020973,-84.368526,13.286102,stress_fragile,gcn_mse 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱分段为 all；晋...
2,analysis.model_regime_window_diagnostic.v1,market_graph_mse,mlp_mixer_mse,41,1,26,26,1,23,15,...,28.986985,all,all,NaN,NaN,-46.223145,-89.871406,14.461503,stress_fragile,market_graph_mse 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱...
6,analysis.model_regime_window_diagnostic.v1,hist_gradient_mse,mlp_mixer_mse,41,1,26,26,1,16,15,...,30.988423,all,all,NaN,NaN,-20.864453,-33.037867,6.088253,stress_fragile,hist_gradient_mse 在压力或下跌分段弱于基准 mlp_mixer_mse，最...
7,analysis.model_regime_window_diagnostic.v1,hypergraph_mse,mlp_mixer_mse,41,1,26,26,1,16,15,...,39.108907,all,all,NaN,NaN,-27.369945,-54.375917,10.402939,stress_fragile,hypergraph_mse 在压力或下跌分段弱于基准 mlp_mixer_mse，最弱分段...


## 中文结论

- 做了什么：把 `research/quantbench_all_model_run_ledger_v1/daily_backtest.csv` 的全模型回测结果接入 `src.analysis.build_market_regime_analysis_report`，用真实 SOXL/SOXX/VIX market panel 重新生成 market-regime/window 模型差异分析。
- 核心指标：覆盖 `2026-05-01` 到 `2026-06-08`，模型数 `32`，daily rows `832`，native/proxy=`28/4`，market regime rows `26`，regime 类型 `4`，窗口数 `8`。
- regime 差异：本轮 market regime 分布为 `high_vol_uptrend=19, stress=5, neutral=1, sideways=1`；最高优先级弱点是 `ridge_ic` 在 `stress` 弱于基准 `sthan_mse`；分段优势候选是 `hawkes_gru_mse` 在 `first_half::high_vol_uptrend`。
- 是否晋升：本轮晋升的是 analysis 框架能力和全模型 market-regime evidence，不晋升任何交易模型。`market_regimes.csv`、`market_regime_counts.csv`、`market_regime_daily_backtest.csv` 已进入 manifest-first analysis report，可供 dashboard/schedule/promotion 继续读取。
- 风险：当前 QuantBench daily backtest 仍有 `8` 个 proxy 模型；market regime window 只有 `26` 个市场日期，短窗结论不能替代更长 walk-forward 验证。压力脆弱模型前列：catboost_rank, extra_trees_rank, logistic_clf, random_forest_clf, ridge_mse。
- 下一步：继续把剩余 proxy 模型改成原生实现，或把 dashboard 的模型详情页改为优先展示本 report 的 market-regime 分段差异。

Sidecar：`analysis_summary.json`、`market_regimes.csv`、`market_regime_counts.csv`、`market_regime_daily_backtest.csv`、`analysis_report/analysis_manifest.json`、`analysis_report/model_review_briefing.csv`、`analysis_report/model_regime_window_diagnostics.csv`。
